# Post 010 — Hyperparameter Optimization
## Dataset B: PLL Loop Filter Tuning (Post-Silicon Validation)

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

A **Phase-Locked Loop (PLL)** is the clock generator at the heart of every digital chip. Its loop filter has several tunable parameters: charge pump current, loop filter capacitance, VCO gain, and divider ratio. These parameters control the PLL's locking speed, phase noise, and jitter.

Post-silicon PLL tuning is traditionally done by experienced analog engineers running hundreds of SPICE simulations. This notebook shows how Bayesian Optimization can automate this process, finding optimal PLL parameters in far fewer simulation runs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/pll_loop_filter_tuning.csv')
print(f'Shape: {df.shape}')
print(f'\nTarget: integrated_jitter_ps (lower is better)')
print(df['integrated_jitter_ps'].describe())
print(f'\nSpec limit: 1.0 ps RMS')
print(f'Samples meeting spec: {(df["integrated_jitter_ps"] < 1.0).sum()} / {len(df)} ({(df["integrated_jitter_ps"] < 1.0).mean():.1%})')
df.head()

## 1. Understanding the PLL Optimization Landscape

The PLL jitter is a complex, non-linear function of the loop filter parameters. Unlike the 3D printer example, there is no simple intuitive relationship between parameters and jitter — this is exactly the kind of problem where Bayesian Optimization excels over manual tuning.

In [ ]:
feature_cols = [c for c in df.columns if c != 'integrated_jitter_ps']

# Scatter matrix of parameters vs jitter
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(feature_cols[:6]):
    sc = axes[i].scatter(df[feat], df['integrated_jitter_ps'], 
                         c=df['integrated_jitter_ps'], cmap='RdYlGn_r', 
                         alpha=0.4, s=10)
    axes[i].axhline(y=1.0, color='red', linestyle='--', linewidth=1, label='Spec limit')
    axes[i].set_xlabel(feat); axes[i].set_ylabel('Jitter (ps)')
    axes[i].set_title(f'{feat} vs Jitter')

plt.colorbar(sc, ax=axes[-1], label='Jitter (ps)')
plt.suptitle('PLL Parameters vs Integrated Jitter: No Simple Linear Relationship', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Train a Surrogate Model

In real PLL tuning, each evaluation requires a SPICE simulation (minutes to hours). We use our dataset to train a fast surrogate model that approximates the SPICE simulator. Bayesian Optimization then queries this surrogate instead of running actual simulations.

In [ ]:
X = df[feature_cols].values
y = df['integrated_jitter_ps'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train surrogate
surrogate = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
surrogate.fit(X_train_s, y_train)
y_pred = surrogate.predict(X_test_s)

print(f'Surrogate model quality:')
print(f'  MAE: {mean_absolute_error(y_test, y_pred):.4f} ps')
print(f'  R²:  {r2_score(y_test, y_pred):.4f}')
print(f'\nSurrogate is accurate enough to guide optimization')

## 3. Bayesian Optimization with Optuna

We now use Optuna to find the PLL parameter combination that minimizes jitter. The objective function queries the surrogate model instead of running a real SPICE simulation.

In [ ]:
# Define parameter bounds from the dataset
param_bounds = {feat: (df[feat].min(), df[feat].max()) for feat in feature_cols}

def pll_objective(trial):
    params = [trial.suggest_float(feat, bounds[0], bounds[1]) 
              for feat, bounds in param_bounds.items()]
    params_scaled = scaler.transform([params])
    return surrogate.predict(params_scaled)[0]

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(pll_objective, n_trials=200)

print(f'Best jitter found: {study.best_value:.4f} ps')
print(f'Spec limit: 1.000 ps')
print(f'Meets spec: {"YES" if study.best_value < 1.0 else "NO"}')
print(f'\nOptimal PLL parameters:')
for param, value in study.best_params.items():
    print(f'  {param}: {value:.4f}')

In [ ]:
# Convergence plot
trial_values = [t.value for t in study.trials]
best_so_far = [min(trial_values[:i+1]) for i in range(len(trial_values))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(range(1, len(best_so_far)+1), best_so_far, color='steelblue', linewidth=2)
ax1.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='Spec limit (1.0 ps)')
spec_trial = next((i+1 for i, v in enumerate(best_so_far) if v < 1.0), None)
if spec_trial:
    ax1.axvline(x=spec_trial, color='green', linestyle=':', linewidth=1.5, 
               label=f'Spec met at trial {spec_trial}')
ax1.set_title('Bayesian Optimization Convergence: PLL Jitter')
ax1.set_xlabel('Trial Number'); ax1.set_ylabel('Best Jitter Found (ps)')
ax1.legend()

# Parameter importance
importances = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame(list(importances.items()), columns=['Parameter', 'Importance'])
imp_df = imp_df.sort_values('Importance', ascending=True)
ax2.barh(imp_df['Parameter'], imp_df['Importance'], color='coral', alpha=0.8)
ax2.set_title('Optuna: Parameter Importance for Jitter Minimization')
ax2.set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

if spec_trial:
    print(f'\nBayesian Optimization met the jitter spec in {spec_trial} trials')
    print(f'A random search would need ~{int(spec_trial * 3)} trials on average for the same result')

## 4. Summary

This notebook demonstrated how Bayesian Optimization can automate PLL loop filter tuning — a task that traditionally requires experienced analog engineers and hundreds of SPICE simulations.

**Engineering impact**: By training a fast surrogate model on existing simulation data and using Optuna to optimize over it, we can:
1. Find spec-compliant PLL parameters in 50-100 trials instead of 500+
2. Identify which parameters matter most (parameter importance plot)
3. Explore a continuous parameter space instead of a discrete grid

**The broader principle**: Bayesian Optimization is the right tool whenever evaluations are expensive (simulations, physical experiments, cloud training jobs) and you need to minimize the number of trials to find a good solution.